In [ ]:

import os
from time import perf_counter as now

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

MMSEG_ROOT = '/mmsegmentation'
os.chdir(MMSEG_ROOT)

from mmseg.apis import init_model
from mmseg.utils import register_all_modules

# ============================================================
# PATHS / CONFIGURATION
# ============================================================

CONFIG_PATH = '/mmsegmentation/zmax_configs/for_test_hasta_26_3/multitask_test_sin_trigger_clean.py'
CKPT_PATH   = '/mmsegmentation/work_dirs/multitask_val_fix_wloss/best_cls_acc_cls_top1_iter_25600.pth'

INPUT_VIDEO = '/0_secuencia_paravideo/1a_secuencia2.mp4'
DEVICE = 'cuda:0'

# Fixed Clockwork
KEYFRAME_K = 30

# GPU-only benchmark
WARMUP_FRAMES = 60
MAX_FRAMES = None            # None = use the entire video
GPU_SAMPLE_EVERY = 1         # 1 = measure all frames
GPU_SAMPLE_OFFSET = 0
PROCESS_SCALE = 1.0
PREPROCESS_USE_PINNED = True
PRED_MASK_DTYPE = np.uint8


# Numeric backend: match the realistic notebook for an apples-to-apples comparison
torch.set_grad_enabled(False)
torch.backends.cudnn.benchmark = False
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

# Export
SAVE_RESULTS = True
RESULTS_DIR = '/mmsegmentation/output/results_clockwork_fixed_1a_k30'
SUMMARY_CSV = os.path.join(RESULTS_DIR, 'summary_gpu_only_global.csv')
DETAILS_CSV = os.path.join(RESULTS_DIR, 'details_gpu_only_per_frame.csv')


In [ ]:

# ============================================================
# UTILITIES
# ============================================================
try:
    from mmseg.structures.dual_task_seg_data_sample import DualTaskSegDataSample as _TestDataSample
except Exception:
    from mmseg.structures import SegDataSample as _TestDataSample


def _label_from_LabelData(lbl):
    import numpy as _np
    if lbl is None:
        return None, None

    x = lbl
    scores = None
    for _ in range(8):
        if x is None:
            return None, scores
        if torch.is_tensor(x):
            return int(x.reshape(-1)[0].detach().cpu().item()), scores
        if isinstance(x, _np.ndarray):
            return int(x.reshape(-1)[0]), scores
        if isinstance(x, (int, float, bool, _np.integer, _np.floating)):
            return int(x), scores
        if isinstance(x, (list, tuple)):
            if len(x) == 0:
                return None, scores
            x = x[0]
            continue

        if hasattr(x, 'item') and callable(getattr(x, 'item')):
            try:
                return int(x.item()), scores
            except Exception:
                pass

        if hasattr(x, 'scores'):
            try:
                s = x.scores
                if torch.is_tensor(s):
                    scores = s.detach().cpu().numpy()
                elif isinstance(s, _np.ndarray):
                    scores = s
                elif isinstance(s, (list, tuple)):
                    scores = _np.asarray(s)
            except Exception:
                pass

        next_x = None
        for k in ('label', 'pred_label', 'data', 'value'):
            if hasattr(x, k):
                next_x = getattr(x, k)
                break
        if next_x is x:
            break
        x = next_x

    return None, scores


class ContextPathClock:
    def __init__(self, bise: torch.nn.Module):
        assert hasattr(bise, 'context_path') and hasattr(bise, 'spatial_path'),                 
        self.context_path = bise.context_path
        self.cache = None
        self.hold = False
        self._orig_forward = self.context_path.forward
        self._install()

    def _install(self):
        @torch.inference_mode()
        def wrapped_forward(x):
            if self.hold and (self.cache is not None):
                return self.cache
            out = self._orig_forward(x)
            if isinstance(out, (list, tuple)):
                self.cache = tuple(o.detach() if torch.is_tensor(o) else o for o in out)
            elif torch.is_tensor(out):
                self.cache = (out.detach(),)
            else:
                self.cache = out
            return out
        self.context_path.forward = wrapped_forward

    def set_hold(self, flag: bool):
        self.hold = bool(flag)

    def invalidate(self):
        self.cache = None
        self.hold = False


class FixedClockScheduler:
    def __init__(self, k=30):
        self.k = max(1, int(k))
        self.next_fire_idx = 0

    def should_fire(self, frame_idx: int) -> bool:
        return int(frame_idx) >= int(self.next_fire_idx)

    def update_after_inference(self, frame_idx: int, did_fire: bool):
        if did_fire:
            self.next_fire_idx = int(frame_idx) + int(self.k)


class NDArrayInferencer:
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.device = next(self.model.parameters()).device

        cfg_dp = model.cfg.model.get('data_preprocessor', {})
        size = cfg_dp.get('size', (512, 512))
        self.input_w = int(size[0])
        self.input_h = int(size[1])
        self.bgr_to_rgb = bool(cfg_dp.get('bgr_to_rgb', True))
        mean = np.array(cfg_dp.get('mean', [123.675, 116.28, 103.53]), dtype=np.float32)
        std  = np.array(cfg_dp.get('std',  [58.395, 57.12, 57.375]), dtype=np.float32)

        self.mean = torch.tensor(mean, device=self.device, dtype=torch.float32).view(1, 3, 1, 1)
        self.std  = torch.tensor(std,  device=self.device, dtype=torch.float32).view(1, 3, 1, 1)

        self._meta = dict(
            ori_shape=(self.input_h, self.input_w),
            img_shape=(self.input_h, self.input_w),
            pad_shape=(self.input_h, self.input_w),
            batch_input_shape=(self.input_h, self.input_w),
            scale_factor=(1.0, 1.0),
            padding_size=[0, 0, 0, 0],
        )

        self.use_cuda = (self.device.type == 'cuda')
        self.use_pinned = bool(PREPROCESS_USE_PINNED and self.use_cuda)

        self._resize_hwc = np.empty((self.input_h, self.input_w, 3), dtype=np.uint8)
        self._cpu_hwc_t = None
        self._cpu_hwc_np = None
        self._gpu_hwc_u8 = None
        self._gpu_chw_f32 = None

        if self.use_pinned:
            self._cpu_hwc_t = torch.empty((1, self.input_h, self.input_w, 3), dtype=torch.uint8, pin_memory=True)
            self._cpu_hwc_np = self._cpu_hwc_t[0].numpy()
            self._gpu_hwc_u8 = torch.empty((1, self.input_h, self.input_w, 3), dtype=torch.uint8, device=self.device)
            self._gpu_chw_f32 = torch.empty((1, 3, self.input_h, self.input_w), dtype=torch.float32, device=self.device)

    def _make_data_sample(self):
        ds = _TestDataSample()
        ds.set_metainfo(self._meta.copy())
        return ds

    def _preprocess(self, img_bgr_nd):
        if (img_bgr_nd.shape[1], img_bgr_nd.shape[0]) != (self.input_w, self.input_h):
            cv2.resize(img_bgr_nd, (self.input_w, self.input_h), dst=self._resize_hwc, interpolation=cv2.INTER_LINEAR)
            img = self._resize_hwc
        else:
            img = img_bgr_nd

        if self.use_pinned:
            np.copyto(self._cpu_hwc_np, img)
            self._gpu_hwc_u8.copy_(self._cpu_hwc_t, non_blocking=True)
            chw_u8 = self._gpu_hwc_u8.permute(0, 3, 1, 2)
            if self.bgr_to_rgb:
                chw_u8 = chw_u8[:, [2, 1, 0], :, :]
            self._gpu_chw_f32.copy_(chw_u8)
            self._gpu_chw_f32.sub_(self.mean).div_(self.std)
            return self._gpu_chw_f32

        if self.bgr_to_rgb:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = np.ascontiguousarray(img.transpose(2, 0, 1))
        tensor = torch.from_numpy(img).unsqueeze(0)
        if self.use_cuda:
            tensor = tensor.to(self.device, non_blocking=True)
        tensor = tensor.float()
        tensor = (tensor - self.mean) / self.std
        return tensor

    @torch.inference_mode()
    def predict_only(self, img_bgr_nd):
        inputs = self._preprocess(img_bgr_nd)
        data_samples = [self._make_data_sample()]
        preds = self.model.predict(inputs, data_samples)
        return preds[0]


def open_video_frames(video_path, max_frames=None):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f'No se pudo abrir el video: {video_path}')
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    limit = total if (max_frames is None) else min(total if total > 0 else max_frames, max_frames)
    idx = 0
    pbar = tqdm(total=(limit if limit else None), desc='Frames', unit='frame')
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if max_frames is not None and idx >= max_frames:
                break
            yield idx, frame
            idx += 1
            pbar.update(1)
    finally:
        pbar.close()
        cap.release()


def maybe_scale_frame(frame):
    if PROCESS_SCALE == 1.0:
        return frame
    return cv2.resize(frame, None, fx=PROCESS_SCALE, fy=PROCESS_SCALE, interpolation=cv2.INTER_AREA)


def build_fixed_pipeline():
    register_all_modules()
    model = init_model(CONFIG_PATH, CKPT_PATH, device=DEVICE)
    model.eval()

    if not (hasattr(model, 'backbone') and hasattr(model.backbone, 'context_path')):
        raise RuntimeError('El modelo no expone backbone.context_path; no se puede aplicar clockwork por features.')

    return {
        'model': model,
        'ctx_clock': ContextPathClock(model.backbone),
        'inferencer': NDArrayInferencer(model),
    }


def reset_fixed_state(pipe):
    pipe['ctx_clock'].invalidate()
    return FixedClockScheduler(k=KEYFRAME_K)



def warmup_fixed(pipe, max_frames=60):
    scheduler = reset_fixed_state(pipe)
    inferencer = pipe['inferencer']
    model = pipe['model']
    for idx, frame in open_video_frames(INPUT_VIDEO, max_frames=max_frames):
        do_fire = scheduler.should_fire(idx)
        pipe['ctx_clock'].set_hold(not do_fire)
        small = maybe_scale_frame(frame)
        inputs = inferencer._preprocess(small)
        data_samples = [inferencer._make_data_sample()]
        _ = model.predict(inputs, data_samples)
        scheduler.update_after_inference(idx, did_fire=do_fire)
        if torch.cuda.is_available():
            torch.cuda.synchronize()


class GPUOnlyProfiler:
    """
    GPU-only measurement comparable with the realistic notebook:
    synchronize -> event_start -> model.predict(inputs, data_samples) -> event_end -> synchronize
    Preprocessing remains OUTSIDE the measured section.
    """
    def __init__(self, sample_every=1, sample_offset=0, use_cuda=True):
        self.sample_every = max(1, int(sample_every))
        self.sample_offset = int(sample_offset)
        self.use_cuda = bool(use_cuda and torch.cuda.is_available())

        self.sum_gpu_ms = 0.0
        self.sum_gpu_adj_ms = 0.0
        self.sum_gpu_stall_ms = 0.0
        self.n_gpu = 0

        self.ev_start = None
        self.ev_end = None
        self.wall_t0 = None
        self._pending_gpu = None

        self.calib_done = False
        self.calib_event_ms = 0.0
        self.calib_stall_ms = 0.0

    def _calibrate(self, iters=30):
        if not self.use_cuda or self.calib_done:
            self.calib_done = True
            return
        torch.cuda.synchronize()
        event_cost = []
        stall_cost = []
        for _ in range(iters):
            e0 = torch.cuda.Event(enable_timing=True)
            e1 = torch.cuda.Event(enable_timing=True)
            t0 = now()
            e0.record()
            e1.record()
            torch.cuda.synchronize()
            t1 = now()
            ms = float(e0.elapsed_time(e1))
            event_cost.append(ms)
            stall_cost.append((t1 - t0) * 1000.0 - ms)
        self.calib_event_ms = float(np.median(event_cost)) if event_cost else 0.0
        self.calib_stall_ms = float(np.median(stall_cost)) if stall_cost else 0.0
        self.calib_done = True

    def want_sample(self, idx):
        return ((int(idx) + self.sample_offset) % self.sample_every) == 0

    def gpu_start(self, idx):
        self._pending_gpu = None
        if not (self.use_cuda and self.want_sample(idx)):
            return False
        if not self.calib_done:
            self._calibrate()
        torch.cuda.synchronize()
        self.ev_start = torch.cuda.Event(enable_timing=True)
        self.ev_end = torch.cuda.Event(enable_timing=True)
        self.wall_t0 = now()
        self.ev_start.record()
        return True

    def gpu_end(self):
        if not (self.use_cuda and self.ev_start is not None and self.ev_end is not None and self.wall_t0 is not None):
            self._pending_gpu = None
            return None
        self.ev_end.record()
        torch.cuda.synchronize()
        wall_t1 = now()

        event_ms = float(self.ev_start.elapsed_time(self.ev_end))
        wall_ms = (wall_t1 - self.wall_t0) * 1000.0
        adj_ms = max(1e-6, event_ms - self.calib_event_ms)
        stall_ms = max(0.0, wall_ms - event_ms)

        self.sum_gpu_ms += event_ms
        self.sum_gpu_adj_ms += adj_ms
        self.sum_gpu_stall_ms += stall_ms
        self.n_gpu += 1

        self._pending_gpu = {
            'gpu_ms': event_ms,
            'gpu_adj_ms': adj_ms,
            'gpu_stall_ms': stall_ms,
        }
        self.ev_start = None
        self.ev_end = None
        self.wall_t0 = None
        return dict(self._pending_gpu)

    def mean_gpu_stall(self):
        return (float(self.sum_gpu_stall_ms) / float(self.n_gpu)) if self.n_gpu > 0 else None


def _summary_global(records, profiler=None):
    vals = np.asarray([r['latency_ms'] for r in records], dtype=np.float64)
    if vals.size == 0:
        raise RuntimeError('No se registraron muestras GPU-only.')

    vals_adj = np.asarray([r['latency_adj_ms'] for r in records if not np.isnan(r['latency_adj_ms'])], dtype=np.float64)
    fires = int(sum(r['phase'] == 'FIRE' for r in records))
    holds = int(sum(r['phase'] == 'HOLD' for r in records))

    rows = []

    def _append(metric_name, arr):
        if arr.size == 0:
            return
        rows.append({
            'method': 'clockwork_fixed_benchmark_comparable',
            'metric': metric_name,
            'n_frames': int(arr.size),
            'mean_ms': float(arr.mean()),
            'fps_from_mean': float(1000.0 / arr.mean()),
            'fires': fires,
            'holds': holds,
            'fire_ratio': float(fires / max(1, len(records))),
            'hold_ratio': float(holds / max(1, len(records))),
            'keyframe_k': int(KEYFRAME_K),
            'video_path': INPUT_VIDEO,
            'config_path': CONFIG_PATH,
            'ckpt_path': CKPT_PATH,
            'median_ms': float(np.median(arr)),
            'p95_ms': float(np.percentile(arr, 95)),
            'min_ms': float(arr.min()),
            'max_ms': float(arr.max()),
            'calib_event_overhead_ms': float(profiler.calib_event_ms) if profiler is not None else np.nan,
            'calib_stall_ref_ms': float(profiler.calib_stall_ms) if profiler is not None else np.nan,
            'runtime_mean_stall_ms': float(profiler.mean_gpu_stall()) if (profiler is not None and profiler.mean_gpu_stall() is not None) else np.nan,
        })

    _append('GPU_ONLY_GLOBAL', vals)
    _append('GPU_ONLY_ADJ_GLOBAL', vals_adj)
    return pd.DataFrame(rows)


def benchmark_fixed_gpu_only(pipe, max_frames=None, sample_every=1, sample_offset=0):
    if not torch.cuda.is_available():
        raise RuntimeError('GPU-only requiere CUDA disponible.')

    scheduler = reset_fixed_state(pipe)
    profiler = GPUOnlyProfiler(sample_every=sample_every, sample_offset=sample_offset, use_cuda=True)
    records = []

    inferencer = pipe['inferencer']
    model = pipe['model']

    for idx, frame in open_video_frames(INPUT_VIDEO, max_frames=max_frames):
        do_fire = scheduler.should_fire(idx)
        pipe['ctx_clock'].set_hold(not do_fire)
        small = maybe_scale_frame(frame)

        # Comparable with the realistic notebook: preprocessing remains outside the GPU-only section.
        inputs = inferencer._preprocess(small)
        data_samples = [inferencer._make_data_sample()]

        do_gpu_timing = profiler.gpu_start(idx)
        preds = model.predict(inputs, data_samples)
        if do_gpu_timing:
            profiler.gpu_end()

        sample = preds[0]

        pred = sample.pred_sem_seg.data
        if pred.ndim == 3 and pred.shape[0] == 1:
            pred = pred[0]
        pred = pred.to(dtype=getattr(torch, str(np.dtype(PRED_MASK_DTYPE).name)))
        pred_mask = pred.detach().cpu().numpy()

        cls_idx = None
        if hasattr(sample, 'pred_label') and sample.pred_label is not None:
            cls_idx, _ = _label_from_LabelData(sample.pred_label)

        scheduler.update_after_inference(idx, did_fire=do_fire)

        pending = profiler._pending_gpu if profiler._pending_gpu is not None else {}
        latency_ms = float(pending.get('gpu_ms', np.nan))
        latency_adj_ms = float(pending.get('gpu_adj_ms', np.nan))
        latency_stall_ms = float(pending.get('gpu_stall_ms', np.nan))
        profiler._pending_gpu = None

        if do_gpu_timing:
            records.append({
                'frame_idx': int(idx),
                'phase': 'FIRE' if do_fire else 'HOLD',
                'latency_ms': latency_ms,
                'latency_adj_ms': latency_adj_ms,
                'latency_stall_ms': latency_stall_ms,
                'cls_idx': cls_idx if cls_idx is not None else np.nan,
                'pred_shape_h': int(pred_mask.shape[0]),
                'pred_shape_w': int(pred_mask.shape[1]),
                'next_fire_idx_after': int(scheduler.next_fire_idx),
            })

    return records, _summary_global(records, profiler=profiler)


In [ ]:

# ============================================================
# MODEL LOADING
# ============================================================
pipe = build_fixed_pipeline()
print('✅ Modelo FIXED cargado.')
print(f'K fijo = {KEYFRAME_K}')
print(f"TF32 matmul activado: {getattr(torch.backends.cuda.matmul, 'allow_tf32', None)}")
print(f"TF32 cuDNN activado: {getattr(torch.backends.cudnn, 'allow_tf32', None)}")
print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark}")


In [ ]:

# ============================================================
# WARM-UP
# ============================================================
print('Warm-up FIXED...')
warmup_fixed(pipe, max_frames=WARMUP_FRAMES)
print('✅ Warm-up listo.')


In [ ]:

# ============================================================
# GPU-ONLY BENCHMARK (COMPARABLE WITH REALISTIC VIDEO K=30)
# ============================================================
records, summary_df = benchmark_fixed_gpu_only(
    pipe,
    max_frames=MAX_FRAMES,
    sample_every=GPU_SAMPLE_EVERY,
    sample_offset=GPU_SAMPLE_OFFSET,
)

details_df = pd.DataFrame(records)

print('=== RESUMEN GLOBAL GPU-ONLY | CLOCKWORK FIJO (COMPARABLE) ===')
display(summary_df)

if SAVE_RESULTS:
    os.makedirs(RESULTS_DIR, exist_ok=True)
    summary_df.to_csv(SUMMARY_CSV, index=False)
    details_df.to_csv(DETAILS_CSV, index=False)
    print('CSV resumen :', SUMMARY_CSV)
    print('CSV detalle :', DETAILS_CSV)
